In [2]:
# ------------------------------------------------------------------ Paso 0: setup
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np
import pyreadstat

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
%load_ext autoreload
%autoreload 2
print(RAIZ)


c:\dev\proyecto1-ehpm-desempleo


In [40]:

# ------------------------------------------------------------------ Paso 1: fijar rutas

RAW       = Path("data/raw")
INTERIM   = Path("data/interim")
PROCESSED = Path("data/processed")
DB_DIR    = Path("db")
DOCS      = Path("docs")
for carpeta in (INTERIM, PROCESSED, DB_DIR, DOCS):
    carpeta.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)

In [41]:
# ------------------------------------------------------------------ Paso 2: ingesta
df, meta = pyreadstat.read_sav(RAW / "hogar_2025.sav")
print("Dimensiones:", df.shape)                         # (24534, 287)
print("Ejemplo de etiqueta:", meta.column_names_to_labels.get("CONDACT"))

Dimensiones: (28714, 342)
Ejemplo de etiqueta: Condicion de actividad


In [42]:
# ------------------------------------------------------------------ Paso 2: inventario
def construir_inventario(df, meta):
    return pd.DataFrame({
        "variable":  df.columns,
        "etiqueta":  [meta.column_names_to_labels.get(c, "") for c in df.columns],
        "dtype":     [str(df[c].dtype) for c in df.columns],
        "pct_na":    (df.isna().mean() * 100).round(1).to_numpy(),
        "n_unique":  [df[c].nunique(dropna=True) for c in df.columns],
    })

inv = construir_inventario(df, meta)
inv.to_csv(DOCS/ "inventario_2025.csv", index=False)

In [5]:
df.head()

,HOGAR,DOMINIO,DEPTO,COR_PRE,NUM_REC,NUM_HOG,V01,V02,V02OTRO,V03,V03OTRO,V04,V04OTRO,V05,V05OTRA,V06,V07,V07OTRO,V08,V08OTRA,V09,V10,V11,V12,H01_1,H01_2,H01_3,H01_4,H01_5,H01_6,...,OCUPAOP,RAMAOS,OCUPAOS,SALMIN,NSALMIN,NUMSALMIN,ACTIVIDA,TOTHRSOP,CATEGOS,ytraop,ytraos,ytrab,yotrf,ytothg,yperhg,quintil,yhpens,yhjubi,yhAlqu,yhOtros,yhayuf,yhayup,yhReme,yheduc,yhbono,ytrabhg,yotrfhg,ymophg,Pobreza,nbi
0,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,5.0,NaN,NaN,12539.68,1.0,2.0,3.0,72.0,99.0,12000.0,NaN,12000.0,NaN,24000.0,4800.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24000.0,NaN,24000.0,2.0,0.0
1,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,5.0,NaN,NaN,12539.68,1.0,2.0,3.0,72.0,99.0,12000.0,NaN,12000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,1.108081e+14,1.0,8.0,80941.0,10.0,1.0,1.0,1.0,,1.0,,3.0,,1.0,,1.0,1.0,,1.0,,6.0,3.0,10000.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [37]:

# ------------------------------------------------------------------ Paso 3: selección
#Definimos un diccionario nombre_estándar → variable del archivo. Trabajar con 23 variables elegidas es mejor
# que con 287. La aserción falla temprano si algún nombre no existe. base es la tabla con la que seguimos.

COLS = {
    "departamento": "DEPTO",   # Departamento del encuestado  
    "edad":      "EDAD",      # años cumplidos (numérica)
    "sexo":      "SEXO",      # 1 Hombre, 2 Mujer
    "area":      "UR",        # 1 Urbana, 2 Rural
    "dominio":   "DOMINIO",   # 5 zonas de estimación
    "educacion": "ED05",      # nivel educativo más alto alcanzado
    "cond_act":  "CONDACT",   # 1 Ocupado, 2 Desocupado, 3 Fuera de la FT
    "tipo_inac": "TIPINAC",   # 2 = Desalentado
    "subocup":   "ACTIVIDA",  # 2 subocupado por tiempo, 3 por ingreso
    "categoria": "CATEGOP",   # categoría ocupacional
    "rama":      "RAMAOP",    # rama de la ocupación principal
    "aguinaldo": "OC620", # recibe aguinaldo
    "catorceavo": "OC621", # recibe catorceavo 
    "vacaciones": "OC622", # recibe bono de vacaciones (Esta y las dos anteriores para estimación de informalidad)
    "horas_principal": "TOTHRSOP",   # solo ocupación principal
    "ytrab":  "ytrab",               # ingreso por trabajo total
    "ytraop": "ytraop",      # Ingreso de la ocupación principal
    "ytraos": "ytraos",              # ingreso de la ocupación secundaria
    "desea_mas_horas": "CA522",      # ¿Hubiera querido trabajar más horas?
    "nsalmin":   "NSALMIN",   # tramo de salario mínimo
    "numsalmin": "NUMSALMIN", # tramo del salario mínimo con definiciones de subocupación
    "salmin":    "SALMIN",    # monto del salario mínimo aplicado
    "factor":    "FACTOR",    # factor de expansión
}
faltan = [v for v in COLS.values() if v not in df.columns]
assert not faltan, f"No están en el archivo: {faltan}"
base = df[list(COLS.values())].rename(columns={v: k for k, v in COLS.items()})
print(base.shape)
base.head(20)

(28714, 23)


,departamento,edad,sexo,area,dominio,educacion,cond_act,tipo_inac,subocup,categoria,rama,aguinaldo,catorceavo,vacaciones,horas_principal,ytrab,ytraop,ytraos,desea_mas_horas,nsalmin,numsalmin,salmin,factor
0,8.0,60.0,1.0,1.0,1.0,4.0,1.0,NaN,3.0,2.0,7.0,2.0,2.0,2.0,72.0,12000.0,12000.0,NaN,9.0,1.0,2.0,12539.68,491.247861
1,8.0,53.0,2.0,1.0,1.0,4.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
2,8.0,30.0,1.0,1.0,1.0,4.0,1.0,NaN,3.0,2.0,7.0,2.0,2.0,2.0,72.0,12000.0,12000.0,NaN,9.0,1.0,2.0,12539.68,491.247861
3,8.0,25.0,2.0,1.0,1.0,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
4,8.0,15.0,2.0,1.0,1.0,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
5,8.0,44.0,1.0,1.0,1.0,5.0,1.0,NaN,1.0,4.0,7.0,NaN,NaN,NaN,48.0,30000.0,30000.0,NaN,2.0,3.0,5.0,12539.68,491.247861
6,8.0,43.0,2.0,1.0,1.0,5.0,1.0,NaN,1.0,1.0,15.0,1.0,1.0,2.0,40.0,16000.0,16000.0,NaN,2.0,2.0,4.0,15115.31,491.247861
7,8.0,16.0,2.0,1.0,1.0,NaN,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
8,8.0,11.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861
9,8.0,10.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,491.247861


In [54]:
base = df[list(COLS.values())].rename(columns={v: k for k, v in COLS.items()})
crudo = base.copy()      # códigos originales, con los nombres estándar
print(base.shape)

(28714, 23)


In [59]:
# ------------------------------------------------------------------ Paso 4: tipado
#Pasamos a numéricas las que lo son, y a categorías con texto las de código. Así las tablas se leen solas y se evitan 
# promedios sin sentido sobre códigos. 

for c in ["edad", "horas_principal", "ytraop", "ytraos", "ytrab", "salmin", "factor"]:
    base[c] = pd.to_numeric(crudo[c], errors="coerce")

etiquetas = {
    "departamento":  {1: "Atlántida", 2: "Colón", 3: "Comayagua", 4: "Copán", 5: "Cortés", 6: "Choluteca",
          7: "El Paraíso", 8: "Francisco Morazán", 9: "Gracias a Dios", 10: "Intibucá",
          11: "Islas de la Bahía", 12: "La Paz", 13: "Lempira", 14: "Ocotepeque", 15: "Olancho",
          16: "Santa Bárbara", 17: "Valle", 18: "Yoro"},
    "sexo":     {1: "Hombre", 2: "Mujer"},
    "area":     {1: "Urbana", 2: "Rural"},
    "dominio":  {1: "Distrito Central", 2: "San Pedro Sula",
                 3: "Ciudades Medianas", 4: "Ciudades Pequeñas", 5: "Rural"},
    "educacion": {1: "Ninguno", 2: "Programa de alfabetización", 3: "Pre-básica (1-3)", 4: "Básica (1-9)", 5: "Media/Diversificado (1-4)",
                  6: "Técnico superior (1-3)", 7: "Superior no universitaria (1-4)", 8: "Superior universitaria (1-8)", 
                  9: "Especialidad (1-3)", 10: "Maestria (1-2)", 11: "Doctorado (1-4)", 99: "No sabe /  No responde"},
    "cond_act": {1: "Ocupado", 2: "Desocupado", 3: "Fuera de la fuerza"},
    "tipo_inac": {1: "Potencialmente Activos", 2: "Desalentados", 3: "Inactivos fuera de la fuerza de trabajo potencial"},
    "subocup": {1: "Sin problemas de empleo", 2: "Subocupados por tiempo de trabajo", 3: "Subocupados por ingreso"},
    "desea_mas_horas": {1: "Sí", 2: "No", 9: "No sabe"},
    "categoria" : {1: "Empleado u obrero público", 2: "Empleado u obrero privado", 3: "Empleado doméstico", 4: "Cuenta propia", 
                    5: "Aprendiz", 6: "Trabajador familiar", 7: "Contratista dependiente", 8: "No declaró categoría"},
    "rama" : {1: "Agricultura, ganadería, silvicultura y pesca", 2: "Explotacion de minas y canteras", 3: "Manufacturera", 
              4: "Suministro de electricidad, gas, vapor y aire acondicionado", 5: "Suministro de agua evacuación de aguas residuales, gestión de desechos y descontaminación", 
              6: "Construcción", 7: "Comercio al por mayor y al por menor reparación de vehículos automotores y motocicletas", 
              8: "Transporte y almacenamiento", 9: "Actividades de alojamiento y de servicio de comidas", 10: "Información y comunicaciones", 
              11: "Actividades financieras y de seguros", 12: "Actividades inmobiliarias", 13: "Actividades profesionales, científicas y técnicas", 
              14: "Actividades de servicios administrativos y de apoyo", 15: "Administración pública y defensa planes de seguridad social de afiliación obligatoria", 
              16: "Enseñanza", 17: "Actividades de atención de la salud humana y de asistencia social", 18: "Actividades artísticas, de entretenimiento y recreativas",
              19: "Otras actividades de servicios", 20: "Actividades de los hogares como empleadores actividades no diferenciadas de los hogares como productores de bienes y ser",
              21: "Actividades de organizaciones y órganos extraterritoriales", 22: "Rama sin especificar", 23: "Busca trabajo por primera vez",
              99: "Ns/Nr"},
    "aguinaldo": {1: "Sí", 2: "No"},
    "catorceavo": {1: "Sí", 2: "No"},
    "vacaciones": {1: "Sí", 2: "No"},
    "nsalmin": {1: "De menos de 1 salario", 2: "De 1 a 2 salarios", 3: "De 2 a 3 salarios",
            4: "De 3 a 4 salarios", 5: "De 4 y más salarios", 99: "No declaran ingresos"},

    "numsalmin": {1: "Menos de 1 salario y trabaja <36 horas", 2: "Menos de 1 salario y trabaja >=36 horas",
              3: "Menos de 1 salario y no declara horas", 4: "De 1 a 2 salarios", 5: "De 2 a 3 salarios",
              6: "De 3 a 4 salarios", 7: "De 4 salarios y más", 9: "No declara ingresos"},
}
for c, mapa in etiquetas.items():
    base[c] = crudo[c].map(mapa).astype("category")
base.dtypes

departamento       category
edad                float64
sexo               category
area               category
dominio            category
educacion          category
cond_act           category
tipo_inac          category
subocup            category
categoria          category
rama               category
aguinaldo          category
catorceavo         category
vacaciones         category
horas_principal     float64
ytrab               float64
ytraop              float64
ytraos              float64
desea_mas_horas    category
nsalmin            category
numsalmin          category
salmin              float64
factor              float64
dtype: object

## 4 Limpieza de faltantes, centinelas y verificación de rangos

In [ ]:
#----------------------------- Revisar el perfil de las variables-------------------------------
def perfil(c):
    orig = COLS[c]
    s = base[c]
    print(c, '|', orig, '|', meta.column_names_to_labels.get(orig))
    print('etiquetas:', meta.variable_value_labels.get(orig))
    print(s.describe())
    print(s.value_counts(dropna=False).head(12))
    if pd.api.types.is_numeric_dtype(s):
        print('mayores:', s.nlargest(5).tolist())
    else:
        print('tipo:', s.dtype)

for c in base.columns:
    perfil(c)
    print('-' * 60)

departamento | DEPTO | Departamento
etiquetas: {1.0: 'Atlántida', 2.0: 'Colón', 3.0: 'Comayagua', 4.0: 'Copán', 5.0: 'Cortes', 6.0: 'Choluteca', 7.0: 'El Paraíso', 8.0: 'Francisco Morazán', 9.0: 'Gracias a Dios', 10.0: 'Intubucá', 11.0: 'Isla de la Bahía', 12.0: 'La Paz', 13.0: 'Lempira', 14.0: 'Ocotepeque', 15.0: 'Olancho', 16.0: 'Santa Bárbara', 17.0: 'Valle', 18.0: 'Yoro'}
count      28714
unique        16
top       Cortés
freq        7899
Name: departamento, dtype: object
departamento
Cortés               7899
Francisco Morazán    5293
Yoro                 1684
El Paraíso           1561
Olancho              1516
Santa Bárbara        1467
Comayagua            1395
Choluteca            1359
Lempira              1125
Copán                1109
Atlántida             998
Colón                 864
Name: count, dtype: int64
tipo: category
------------------------------------------------------------
edad | EDAD | Edad
etiquetas: None
count    28438.000000
mean        32.416590
std         2

In [46]:
pd.crosstab(df['TIPINAC'], base['cond_act'], dropna=False)

cond_act,Desocupado,Fuera de la fuerza,Ocupado
TIPINAC,,,
NaN,593,0,11616
1.0,0,74,0
2.0,0,362,0
3.0,0,8308,0


In [60]:
# 1) ¿el mapeo creó vacíos que antes no existían?
for c, orig in COLS.items():
    antes, despues = df[orig].isna().sum(), base[c].isna().sum()
    if antes != despues:
        print(c, 'vacíos antes:', antes, 'después:', despues)

# 2) ¿hay códigos en los datos sin etiqueta en el archivo?
for c, orig in COLS.items():
    etq = meta.variable_value_labels.get(orig)
    if etq:
        sin_etiqueta = set(df[orig].dropna().unique()) - set(etq)
        if sin_etiqueta:
            print(c, 'códigos sin etiqueta:', sin_etiqueta)

tipo_inac códigos sin etiqueta: {np.float64(3.0)}


In [61]:
for c in ["edad", "horas_principal", "ytraop", "ytraos", "ytrab", "salmin", "factor"]:
    base[c] = pd.to_numeric(crudo[c], errors="coerce")

for c, mapa in etiquetas.items():
    orden = pd.CategoricalDtype(list(mapa.values()))
    base[c] = crudo[c].map(mapa).astype(orden)

In [62]:
for c in base.columns:
    antes, despues = crudo[c].isna().sum(), base[c].isna().sum()
    if antes != despues:
        print(f"{c}: vacíos antes {antes} → después {despues}")

In [64]:
for c in etiquetas:
    en_datos = {float(x) for x in crudo[c].dropna().unique()}
    en_meta  = {float(x) for x in meta.variable_value_labels.get(COLS[c], {})}
    en_dicc  = {float(x) for x in etiquetas[c]}
    if en_datos - en_dicc:
        print(f"{c}: sin etiqueta en TU diccionario {sorted(en_datos - en_dicc)}")
    if en_datos - en_meta:
        print(f"{c}: sin etiqueta en el ARCHIVO {sorted(en_datos - en_meta)}")

tipo_inac: sin etiqueta en el ARCHIVO [3.0]


In [11]:
#print(base["horas"].unique())          # array con los valores distintos
#print(base["horas"].value_counts())   # cada valor con su frecuencia, de mayor a menor
print(base["nsalmin"].unique()) 
#print(base["nsalmin"].value_counts())  
#print(base["salmin"].unique())  
#print(base["salmin"].value_counts()) 
print(df["NUMSALMIN"].unique())  
#print(df["NUMSALMIN"].value_counts()) 

[ 1. nan  3.  2. 99.  4.  5.]
[ 2. nan  5.  4.  9.  1.  6.  7.]
